<a href="https://colab.research.google.com/github/xl-24/sal_study/blob/main/readback_check_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code based on claude. This is workable version of code (4/25/2026). With using G5 GPU (faster than A100, the average processng time is 2-3 sec.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, time

In [ ]:
# Cell 2 — Load model (A100-optimized, no quantization, no compile)
MODEL_ID = "google/gemma-4-e4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cuda",          # explicit single-GPU, avoids device_map overhead
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa" # Gemma 4 may not support flash_attention_2 yet
)
model.eval()
model.config.use_cache = True

# Skip torch.compile for now — unreliable with variable-length inputs
# Verify GPU and memory
print(f"Device: {next(model.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("Model loaded ✓")

In [ ]:
# Cell 3 — System prompt & inference

SYSTEM_PROMPT = """You are an expert ATC (Air Traffic Control) communications analyst.
Your job is to compare a controller's instruction with a pilot's readback and identify readback errors.

## Leniency rules — do NOT flag these as errors:
- Informal altitude phrasing: "up to 4000" / "out of 4000" / "leaving 4000" = "climb/descend maintain 4000"
- Equivalent direction words: "going up to" / "climbing to" = "climb maintain", "coming down to" / "descending to" = "descend maintain"
- Callsign abbreviations: last 3 characters of a callsign are acceptable (e.g., "10BW" or "Bravo Whiskey" = "Cessna 310BW")
- Callsign type omission: dropping aircraft type ("310BW" = "Cessna 310BW") is acceptable
- Filler/connector words: ignore "roger", "wilco", "affirmative", "and", "with you", "out of", positional reports like "passing through"
- Word/digit order variation: readback elements in a different order than the instruction is acceptable
- Numeric format equivalence: "four thousand" = "4000" = "4 thousand", "one niner zero" = "190"
- Rounding/spoken variants: "flight level two five zero" = "FL250" = "two five zero"

## Only flag true safety-critical discrepancies:
- Wrong altitude, heading, squawk code, or frequency VALUE
- Wrong callsign identity (different aircraft entirely, not just abbreviated)
- Missing safety-critical element with NO equivalent phrasing (e.g., cleared altitude never mentioned in any form)
- Transposed digits that change a value (e.g., "4500" said as "5400")

## Error categories:
- OMISSION: pilot completely omitted a required safety element (no equivalent phrasing present)
- SUBSTITUTION: pilot stated a different value than instructed
- TRANSPOSITION: pilot swapped digits/words changing the meaning
- ADDITION: pilot added a clearance or instruction not given (only flag if it could cause confusion)
- CORRECT: readback conveys the same intent and all safety-critical values match

Respond ONLY with valid JSON in this exact format:
{
  "status": "CORRECT" | "ERROR",
  "errors": [
    {
      "type": "OMISSION|SUBSTITUTION|TRANSPOSITION|ADDITION",
      "element": "what part is affected (e.g., altitude, callsign, heading)",
      "controller_said": "exact value from instruction",
      "pilot_said": "exact value from readback or null if omitted",
      "severity": "HIGH|MEDIUM|LOW"
    }
  ],
  "summary": "one-sentence plain English explanation",
  "safe_to_proceed": true | false
}"""


In [ ]:
# Cell 4 — Stopping criteria (fresh instance per call to reset brace counter)
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnCompleteJSON(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.open_id  = tokenizer.encode("{", add_special_tokens=False)[0]
        self.close_id = tokenizer.encode("}", add_special_tokens=False)[0]
        self.depth   = 0
        self.started = False

    def __call__(self, input_ids, scores, **kwargs):
        last_token = input_ids[0, -1].item()
        if last_token == self.open_id:
            self.depth += 1
            self.started = True
        elif last_token == self.close_id:
            self.depth -= 1
        # Stop only when all braces are balanced AND at least one was opened
        return self.started and self.depth == 0

def build_stopping_criteria(tokenizer):
    # Called inside check_readback so counter resets on every inference call
    return StoppingCriteriaList([StopOnCompleteJSON(tokenizer)])

In [ ]:
# Cell 5 — Inference with fixed-length padding for stable latency
def check_readback(controller_instruction: str, pilot_readback: str) -> dict:
    user_message = f"""Controller instruction: "{controller_instruction}"
Pilot readback: "{pilot_readback}"

Analyze this readback for errors."""

    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_message}"}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding="max_length",
        max_length=1024,
        truncation=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            stopping_criteria=build_stopping_criteria(tokenizer),  # fresh per call
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    clean = raw_response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        print(f"[DEBUG] Raw output: {repr(raw_response)}")
        return {"status": "PARSE_ERROR", "raw": raw_response, "summary": "Could not parse model output"}

In [ ]:
# Cell 6 — Warmup (run this before your test loop)
print("Warming up GPU...")
torch.cuda.synchronize()
_ = check_readback("warmup", "warmup")
torch.cuda.synchronize()
print("Warmup done ✓")

In [ ]:
# Cell 7 — Pretty printer

def display_result(controller: str, pilot: str, result: dict):
    status_icon = "✅" if result.get("status") == "CORRECT" else "🚨"
    proceed_icon = "✅" if result.get("safe_to_proceed") else "⛔"

    print(f"\n{'='*60}")
    print(f"📡 Controller: {controller}")
    print(f"🎙️  Pilot:      {pilot}")
    print(f"{'─'*60}")
    print(f"Status:          {status_icon} {result.get('status', 'UNKNOWN')}")
    print(f"Safe to proceed: {proceed_icon}")
    print(f"Summary:         {result.get('summary', '')}")

    errors = result.get("errors", [])
    if errors:
        print(f"\nErrors found ({len(errors)}):")
        for i, e in enumerate(errors, 1):
            sev_icon = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🔵"}.get(e.get("severity",""), "⚪")
            print(f"  {i}. {sev_icon} [{e['type']}] — {e['element']}")
            print(f"     Controller said: '{e.get('controller_said', 'N/A')}'")
            print(f"     Pilot said:      '{e.get('pilot_said', 'OMITTED')}'")
    print(f"{'='*60}")

In [ ]:
# Cell 6 — Test with your example + more cases

test_pairs = [
    # Your original example
    (
        "Cessna 310BW climb maintain four thousand",
        "out of 4 thousand Bravo Whiskey"
    ),
    # Altitude substitution
    (
        "United 472 descend and maintain flight level 250",
        "descend and maintain flight level 350 United 472"
    ),
    # Correct readback
    (
        "Delta 881 turn right heading 090",
        "right heading 090 Delta 881"
    ),
    # Omission of callsign
    (
        "Southwest 1234 contact approach 119.5",
        "contact approach 119.5"
    ),
    # Transposition
    (
        "N5437Q squawk 4573",
        "squawk 4753 N5437Q"
    ),
]

results = []
import time

for i, (controller, pilot) in enumerate(test_pairs, 1):
    start = time.perf_counter()
    result = check_readback(controller, pilot)
    elapsed = time.perf_counter() - start

    display_result(controller, pilot, result)
    print(f"⏱  Processing time: {elapsed:.2f}s")
    results.append({
        "controller": controller,
        "pilot": pilot,
        "analysis": result,
        "processing_time_s": round(elapsed, 2)
    })

avg = sum(r["processing_time_s"] for r in results) / len(results)
print(f"\n📊 Average processing time across {len(results)} checks: {avg:.2f}s")